# Lesson 8.4: Multiple Regression: Many Predictors at Once

Predict house price from square feet and bedrooms together, and see why the bedrooms effect disappears once size is held fixed, using the lesson's ten houses.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A real-estate agent has ten houses with their size (square feet), number of bedrooms and sale price (in thousands of dollars). A client asks: if I add a bedroom, how much more is the house worth? Bigger houses have more bedrooms and higher prices, so is the bedroom doing the work, or is it just standing in for size?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the ten houses

Square feet, bedrooms and price (in thousands of dollars). Bigger houses tend to have more bedrooms, so the two predictors are tangled together. That is the whole point of this lab.

In [1]:
import pandas as pd  # Load pandas, which holds data in tables (DataFrames).
import statsmodels.formula.api as smf  # Load statsmodels' formula interface, which lets us write models like 'price ~ sqft'.

df = pd.DataFrame({  # Build a table from a dictionary: each name becomes a column and each list holds that column's ten values.
    'sqft': [1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800],  # Square feet of each house, from 1,000 to 2,800.
    'bedrooms': [2, 2, 3, 3, 4, 4, 4, 5, 5, 6],  # Number of bedrooms in each house, in the same order.
    'price': [250, 280, 310, 350, 390, 420, 450, 480, 520, 550]})  # Sale price in thousands of dollars, in the same order.

print(df.corr().round(3))  # Print the correlation of every pair of columns, rounded to 3 decimals, to see how tangled the variables are.

           sqft  bedrooms  price
sqft      1.000     0.976  0.999
bedrooms  0.976     1.000  0.975
price     0.999     0.975  1.000


**What this means.** Price is almost perfectly linked to size (0.999) and strongly to bedrooms (0.975). But size and bedrooms are also strongly linked to each other (0.976): bigger houses simply have more bedrooms. When predictors are tangled like this, it is hard to say which one is doing the work.

## Step 2: Bedrooms alone

On its own, bedrooms seems to matter a lot: each extra bedroom goes with a large jump in price. But bedrooms may just be standing in for size.

In [2]:
alone = smf.ols('price ~ bedrooms', data=df).fit()  # Fit a model with bedrooms as the only predictor. 'price ~ bedrooms' reads 'price explained by bedrooms'.

print(f'bedrooms alone: {alone.params["bedrooms"]:.1f} thousand per bedroom')  # Print its slope: the average price change per extra bedroom if we ignore size.
print(f'R squared = {alone.rsquared:.3f}')  # Print R squared, the share of price variation that bedrooms alone explain.

bedrooms alone: 75.6 thousand per bedroom
R squared = 0.952


**What this means.** On its own, bedrooms looks powerful: each extra bedroom goes with 75.6 thousand dollars more, and R squared is 0.952. That is the figure a simple regression would give the client. But it may be size in disguise.

## Step 3: Both predictors together

Adding sqft to the formula with + fits one model with two predictors. Each coefficient is the change in price for one more unit of that predictor while the other is held fixed.

In [3]:
both = smf.ols('price ~ sqft + bedrooms', data=df).fit()  # Put both predictors in one model. The + in the formula adds sqft next to bedrooms.

print(both.params.round(4))  # Print the intercept and the two coefficients, rounded to 4 decimals.
print()  # Print an empty line to separate the two outputs.
print(both.pvalues.round(4))  # Print each coefficient's p-value. It tests whether that coefficient could really be 0, given the other predictor is in the model.

Intercept    80.0645
sqft          0.1668
bedrooms      0.8065
dtype: float64

Intercept    0.0000
sqft         0.0000
bedrooms     0.8717
dtype: float64


**What this means.** With both in the model, sqft has a coefficient of 0.1668: each extra square foot adds about 0.167 thousand dollars ($167), holding bedrooms fixed. Bedrooms drops to 0.8065 (about $806) with p = 0.8717, far above 0.05. Once size is accounted for, bedrooms adds essentially nothing.

## Step 4: Read the fit and what 'held fixed' means

The sqft coefficient compares two houses with the same number of bedrooms and different sizes. The bedrooms coefficient compares houses of the same size with different bedrooms. R squared says how much of the price variation the pair explains.

In [4]:
print(f'R squared          = {both.rsquared:.4f}')  # R squared for the two-predictor model: how much of the price variation they explain together.
print(f'adjusted R squared = {both.rsquared_adj:.4f}')  # Adjusted R squared, with a small penalty for each predictor so that adding a useless one does not look like progress.
print(f'overall F = {both.fvalue:.1f}, p = {both.f_pvalue:.2e}')  # The overall F test asks whether the predictors together explain anything. .2e prints the p-value in scientific notation.

new = pd.DataFrame({'sqft': [1500], 'bedrooms': [3]})  # Describe a new house: 1,500 square feet and 3 bedrooms, as a one-row table with the same column names.
print(f'predicted price, 1500 sqft, 3 bed = {both.predict(new)[0]:.1f}')  # Put the new house through the fitted model to predict its price.

R squared          = 0.9987
adjusted R squared = 0.9983
overall F = 2688.9, p = 7.92e-11
predicted price, 1500 sqft, 3 bed = 332.6


**What this means.** Together the two predictors explain 99.87% of the price variation (adjusted 99.83%), and the overall F test is overwhelming (F = 2688.9, p = 7.92e-11). Most of that comes from sqft. The model predicts 332.6 thousand dollars for 1,500 sq ft and 3 bedrooms. These are correlations from ten houses, not proof of what causes price.

## So, what is the answer?

Ignoring size, each bedroom seemed to add $75,600. Holding size fixed, a bedroom adds almost nothing (0.8 thousand dollars, p = 0.87), while each extra square foot adds about $167, so the bedrooms were mostly standing in for size. Caution: size and bedrooms are so tangled (r = 0.98) that ten houses cannot cleanly separate them, and this shows association, not what a renovation would do.

## Now you try

A courier company records, for ten deliveries, the distance driven (km), the number of stops and the total delivery time in minutes. Longer trips usually have more stops. How much time does each extra km add, and each extra stop, when the other is held fixed?

**Data:** distance = [2, 4, 5, 7, 8, 10, 12, 13, 15, 18]
stops = [1, 2, 2, 3, 3, 4, 4, 5, 6, 7]
minutes = [14, 22, 25, 33, 36, 44, 50, 56, 66, 80]

1. Check the correlations between the three variables.
2. Fit minutes on stops alone and note the coefficient.
3. Fit minutes on distance and stops together; report both coefficients, their p-values and R squared.
4. Predict the time for 10 km and 4 stops, and explain how the stops coefficient changed.

*Hint:* Build a DataFrame and use smf.ols('minutes ~ distance + stops', data=df).fit(). Compare the stops coefficient with and without distance in the model.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.DataFrame({'distance': [2, 4, 5, 7, 8, 10, 12, 13, 15, 18],
                   'stops': [1, 2, 2, 3, 3, 4, 4, 5, 6, 7],
                   'minutes': [14, 22, 25, 33, 36, 44, 50, 56, 66, 80]})
print(df.corr().round(3))
print(smf.ols('minutes ~ stops', data=df).fit().params.round(2).to_dict())
m = smf.ols('minutes ~ distance + stops', data=df).fit()
print(m.params.round(3).to_dict())
print(m.pvalues.round(4).to_dict())
print(f'R squared = {m.rsquared:.4f}, adjusted = {m.rsquared_adj:.4f}')
print(m.predict(pd.DataFrame({'distance': [10], 'stops': [4]})).round(1).tolist())

          distance  stops  minutes
distance     1.000  0.990    0.997
stops        0.990  1.000    0.995
minutes      0.997  0.995    1.000
{'Intercept': 2.17, 'stops': 10.93}
{'Intercept': 3.425, 'distance': 2.352, 'stops': 4.613}
{'Intercept': 0.0063, 'distance': 0.0029, 'stops': 0.0143}
R squared = 0.9975, adjusted = 0.9968
[45.4]


**Answer.** Distance and stops are tightly linked (0.990). Stops alone give 10.93 minutes per stop. With both in the model, each km adds 2.352 minutes (p = 0.0029) and each stop adds 4.613 minutes (p = 0.0143), holding the other fixed. The stops effect shrank from 10.93 to 4.61 because stops were partly standing in for distance. R squared = 0.9975 (adjusted 0.9968). A 10 km, 4 stop delivery is predicted at 45.4 minutes.

## The same question in R, SPSS and Stata

**R**

```r
sqft <- c(1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800)  # Square feet of the ten houses.
bedrooms <- c(2, 2, 3, 3, 4, 4, 4, 5, 5, 6)  # Number of bedrooms of the ten houses, in the same order.
price <- c(250, 280, 310, 350, 390, 420, 450, 480, 520, 550)  # Sale price in thousands of dollars, in the same order.
summary(lm(price ~ bedrooms))  # Fit and summarise a model with bedrooms alone. lm(price ~ bedrooms) means 'price explained by bedrooms'.
fit <- lm(price ~ sqft + bedrooms)  # Fit the model with two predictors; the + adds sqft. We save it as fit so we can reuse it.
summary(fit)  # Print the full summary: coefficients with p-values, R-squared, adjusted R-squared and the overall F test.
predict(fit, data.frame(sqft = 1500, bedrooms = 3))  # Predict the price of a new house with 1,500 sq ft and 3 bedrooms, supplied as a data frame.
```

How to read it: In the first summary, bedrooms alone has Estimate 75.6. In the second, sqft has Estimate 0.1668 with a tiny p-value, while bedrooms has Estimate 0.806 and Pr(>|t|) 0.872: not significant once size is in. Multiple R-squared is 0.9987 and the F-statistic is 2689. predict gives 332.6.

**SPSS** (Analyze > Regression > Linear (Dependent: price, Independents: sqft and bedrooms))

```
DATA LIST FREE / sqft bedrooms price.
BEGIN DATA
1000 2 250  1200 2 280  1400 3 310  1600 3 350  1800 4 390
2000 4 420  2200 4 450  2400 5 480  2600 5 520  2800 6 550
END DATA.
REGRESSION
  /STATISTICS COEFF R ANOVA
  /DEPENDENT price
  /METHOD=ENTER sqft bedrooms.
```

How to read it: In the Coefficients table, read B and Sig. for each row: sqft B = .167, Sig. = .000; bedrooms B = .806, Sig. = .872 (not significant). Model Summary: R Square = .999, Adjusted R Square = .998. ANOVA: F = 2688.9, Sig. = .000. To see bedrooms alone, run the model again with only bedrooms.

**Stata**

```
clear
set obs 10
generate sqft = 800 + 200 * _n
generate bedrooms = real(word("2 2 3 3 4 4 4 5 5 6", _n))
generate price = real(word("250 280 310 350 390 420 450 480 520 550", _n))
regress price bedrooms
regress price sqft bedrooms
display _b[_cons] + _b[sqft] * 1500 + _b[bedrooms] * 3
```

How to read it: In the first regress, bedrooms coef is 75.6. In the second, sqft coef .1668 has P>|t| 0.000 while bedrooms coef .8065 has P>|t| 0.872. Also read R-squared 0.9987, Adj R-squared 0.9983 and F(2, 7) = 2688.9. The display shows 332.6, the predicted price.